# 小规模 QUBO Solver 验证

这本 notebook 使用仓库的 `qubo.v1` / `qubo-result.v1` 契约，对小规模 solver 做可重复验证。

验证顺序刻意分成四层：

1. 先验证输入 QUBO 的结构契约；
2. 用独立穷举建立 oracle，不依赖任何 solver；
3. 分别运行 Exact 与 QAOA solver，并验证结果契约；
4. 比较 sample、energy、status 和 optimality gap。

> QAOA 是近似算法，正确行为是返回 `feasible`，而不是宣称 `optimal`。

## 1. 环境与导入

请在 VS Code/Jupyter 中选择项目的 `.venv` kernel。下面的代码会自动向上寻找仓库根目录，因此从仓库根目录或 `notebooks/` 目录启动都可以。

In [ ]:
import copy
import json
import sys
from fractions import Fraction
from itertools import product
from pathlib import Path


def find_project_root(start):
    """Find the repository without assuming the notebook working directory."""
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "lib").is_dir() and (candidate / "problem").is_dir():
            return candidate
    raise RuntimeError("Could not locate the QSolutionData repository root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.contracts import validate_qubo, validate_qubo_result
from lib.solvers.qubo import ExactQuboSolver, QaoaQuboSolver

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

## 2. 定义一个 `qubo.v1`

这里构造三个二进制变量的最小化问题：每个变量有不同 reward，同时选择多个变量会产生 pair penalty。

仓库采用上三角稀疏项：`[i, j, coefficient]` 表示 $Q_{ij}x_ix_j$，offset 单独保存。

In [ ]:
qubo = {
    "schema": "qubo.v1",
    "problem_id": "notebook-small-qubo",
    "sense": "minimize",
    "num_variables": 3,
    "variable_names": ["asset_alpha", "asset_beta", "asset_gamma"],
    "offset": 1.0,
    "terms": [
        [0, 0, -3.0],
        [1, 1, -2.0],
        [2, 2, -1.0],
        [0, 1, 4.0],
        [0, 2, 2.5],
        [1, 2, 2.0],
    ],
    "metadata": {
        "purpose": "small deterministic solver validation",
        "expected_unique_optimum": [1, 0, 0],
    },
}
qubo_snapshot = copy.deepcopy(qubo)

validate_qubo(qubo)
print(json.dumps(qubo, indent=2, ensure_ascii=False))

## 3. 独立穷举 oracle

下面的 evaluator 是 notebook 自己写的，并使用 `Fraction` 累加。它不调用 solver，也不调用 solver 内部 energy helper，因此可以作为独立交叉检查。

In [ ]:
def independent_qubo_energy(problem, sample):
    """Evaluate one canonical QUBO sample with exact JSON-number arithmetic."""
    energy = Fraction(problem["offset"])
    for left, right, coefficient in problem["terms"]:
        energy += Fraction(coefficient) * sample[left] * sample[right]
    return energy


def public_json_number(exact_value):
    """Mirror the public contract's integral-or-float JSON representation."""
    if exact_value.denominator == 1:
        return exact_value.numerator
    return float(exact_value)


truth_table = []
for bits in product((0, 1), repeat=qubo["num_variables"]):
    sample = list(bits)
    truth_table.append({
        "sample": sample,
        "energy_exact": independent_qubo_energy(qubo, sample),
    })

truth_table.sort(key=lambda row: (row["energy_exact"], row["sample"]))
oracle_sample = truth_table[0]["sample"]
oracle_energy = truth_table[0]["energy_exact"]

for row in truth_table:
    print(row["sample"], "->", float(row["energy_exact"]))

assert oracle_sample == [1, 0, 0]
assert oracle_energy == Fraction(-2)
print("Oracle optimum:", oracle_sample, "energy =", float(oracle_energy))

## 4. Exact solver

`ExactQuboSolver` 会枚举全部 $2^n$ 个 assignment。它适合作为小问题正确性 oracle，不适合大规模生产问题。

In [ ]:
exact_solver = ExactQuboSolver()
exact_config = {"max_variables": 10}
exact_config_snapshot = copy.deepcopy(exact_config)
exact_result = exact_solver.solve(
    qubo,
    config=exact_config,
)
validate_qubo_result(qubo, exact_result)

assert exact_result["status"] == "optimal"
assert exact_result["termination_reason"] == "search_exhausted"
exact_energy_exact = independent_qubo_energy(
    qubo,
    exact_result["best_sample"],
)
assert exact_result["best_sample"] == oracle_sample
assert exact_energy_exact == oracle_energy
assert exact_result["best_energy"] == public_json_number(exact_energy_exact)
assert exact_result["metrics"]["search_space_exhausted"] is True
assert exact_result["metrics"]["candidates_evaluated"] == 1 << qubo["num_variables"]
assert qubo == qubo_snapshot
assert exact_config == exact_config_snapshot

print(json.dumps(exact_result, indent=2, ensure_ascii=False))

## 5. QAOA solver

这里使用 NumPy statevector 参考实现。固定 seed 保证实验可重复；`shots` 模式会在观测到的状态中返回最低能量样本。

In [ ]:
qaoa_config = {
    "layers": 2,
    "optimizer_iterations": 20,
    "restarts": 4,
    "shots": 4096,
    "seed": 7,
    "max_variables": 10,
}

qaoa_config_snapshot = copy.deepcopy(qaoa_config)
qaoa_solver = QaoaQuboSolver()
qaoa_result = qaoa_solver.solve(qubo, config=qaoa_config)
qaoa_repeat = qaoa_solver.solve(qubo, config=qaoa_config)
validate_qubo_result(qubo, qaoa_result)

assert qaoa_result["status"] == "feasible"
assert qaoa_result["best_sample"] is not None
qaoa_energy_exact = independent_qubo_energy(
    qubo,
    qaoa_result["best_sample"],
)
assert qaoa_energy_exact >= oracle_energy
assert qaoa_result["best_energy"] == public_json_number(qaoa_energy_exact)
for field in ("best_sample", "best_energy", "metrics", "metadata"):
    assert qaoa_result[field] == qaoa_repeat[field]
assert qubo == qubo_snapshot
assert qaoa_config == qaoa_config_snapshot

print(json.dumps(qaoa_result, indent=2, ensure_ascii=False))

## 6. 对比结果

Exact 必须与 oracle 完全一致。QAOA 的 gap 必须非负；它可能命中最优解，但依然只声明 `feasible`，因为采样本身不构成全局最优性证明。

In [ ]:
comparison = [
    {
        "solver": "independent enumeration",
        "status": "proved optimal",
        "sample": oracle_sample,
        "energy": float(oracle_energy),
        "gap": 0.0,
    },
    {
        "solver": exact_result["solver"]["name"],
        "status": exact_result["status"],
        "sample": exact_result["best_sample"],
        "energy": exact_result["best_energy"],
        "gap": float(exact_energy_exact - oracle_energy),
    },
    {
        "solver": qaoa_result["solver"]["name"],
        "status": qaoa_result["status"],
        "sample": qaoa_result["best_sample"],
        "energy": qaoa_result["best_energy"],
        "gap": float(qaoa_energy_exact - oracle_energy),
    },
]

for row in comparison:
    print(row)

assert comparison[1]["gap"] == 0.0
assert comparison[2]["gap"] >= 0.0

## 7. 负例：输入和输出都不能绕过契约

先给 QUBO 添加重复稀疏项，再篡改 solver 返回的 energy。两者都必须在公共契约边界被拒绝。

In [ ]:
invalid_qubo = copy.deepcopy(qubo)
invalid_qubo["terms"].append([0, 0, -3.0])

validation_error = None
try:
    validate_qubo(invalid_qubo)
except (TypeError, ValueError) as error:
    validation_error = str(error)

assert validation_error is not None
print("Expected input validation error:", validation_error)

tampered_result = copy.deepcopy(qaoa_result)
tampered_result["best_energy"] += 1
result_error = None
try:
    validate_qubo_result(qubo, tampered_result)
except (TypeError, ValueError) as error:
    result_error = str(error)

assert result_error is not None
print("Expected result validation error:", result_error)

## 结论

这套小规模校验确认了：

- `qubo.v1` 在运行前通过严格契约；
- Exact solver 与独立穷举 oracle 完全一致；
- QAOA 返回合法、可重算的 `qubo-result.v1`，但保守地保持 `feasible`；
- 坏的稀疏项会在 solver 边界之前被拒绝。

复用这本模板时，请替换 `qubo`，并同步更新 fixture 专属的预期 sample/energy 断言；候选数量等通用断言会从模型维度自动推导。